In [ ]:
# Fine-tune v2 into a conversation model, v3. In the chat page v2 sees only the last 2-3 exchanges (256
# tokens). This notebook starts from v2's weights on the Hugging Face Hub, grows its context to 1024 tokens
# (about 13 exchanges), and trains it on dialogue from the plays in the chat page's format, with the loss
# on the answering character's speeches only.
# What it won't fix: how much a reply depends on the line it answers. Shakespeare's real reply is only ~2%
# more likely after the line it answered than after a random line, and a fine-tune barely moves that: the
# answers were already in v2's training text, so this is the same information reweighted. That takes a
# bigger model. The line gain is still measured below, to keep track of it.
# Imports, seeds, and device: CUDA on Colab (Runtime -> Change runtime type -> T4 GPU), MPS on Apple Silicon, otherwise CPU.
import contextlib
import json
import math
import random
import re
import time
import urllib.request
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from torch.nn import functional as F

torch.manual_seed(1337)
random.seed(1337)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# The starting point: v2 from the Hugging Face Hub, pinned to a commit so reruns start from the same weights.
# Its npz (the file the inference server loads) holds the weights, the config and the tokenizer.
V2_URL = "https://huggingface.co/hr5ht/tinygpt-shakespeare-v2/resolve/1840ceb7a940ea483a895831b125dc8afc5ef791/stories.npz"
v2_path = Path("data/stories.npz")
if not v2_path.exists():
    v2_path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(V2_URL, v2_path)

with np.load(v2_path) as f:
    v2_weights = {k: torch.from_numpy(f[k]) for k in f.files if k not in ("config", "tokenizer")}
    v2_config = json.loads(str(f["config"]))
    tokenizer_json = str(f["tokenizer"])
tok = Tokenizer.from_str(tokenizer_json)
EOT = tok.token_to_id("<|endoftext|>")


def encode(s):
    return tok.encode(s).ids


def decode(ids):
    return tok.decode(ids)


print(v2_config)
print(f"{sum(w.numel() for w in v2_weights.values()) / 1e6:.2f}M params, {tok.get_vocab_size()} word pieces")

In [ ]:
# The complete works of Shakespeare (Project Gutenberg #100), prepared as in model.ipynb: only the text
# between Gutenberg's START/END markers, curly quotes turned into straight ones.
URL = "https://www.gutenberg.org/cache/epub/100/pg100.txt"
path = Path("data/shakespeare.txt")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, path)

raw = path.read_text(encoding="utf-8-sig")
start = raw.index("\n", raw.index("*** START OF")) + 1
end = raw.index("*** END OF")
text = raw[start:end].replace("\r\n", "\n").strip()
text = text.translate(str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'}))
print(f"{len(text):,} characters")

In [ ]:
# Split the plays into scenes, and each scene into (speaker, speech). A speech starts at a line like "ROMEO."
# and runs until the next speaker. Stage directions (indented lines, [_Aside._]) are dropped, and verse that
# resumes after a stage direction stays with its speaker. Headings (ACT, SCENE, play titles) end a scene.
# The last 10% of scenes are held out, which is roughly the last 10% of the text, as in model.ipynb.
SPEAKER = re.compile(r"[A-Z][A-Z' -]*[A-Z]\.")  # "ROMEO.", "LADY MACBETH.", "FIRST CITIZEN."


def clean(lines):
    lines = (re.sub(r"\[[^\]]*\]?", "", line).replace("_", "").strip() for line in lines if not line.startswith(" "))
    return "\n".join(line for line in lines if line)


def split_scenes(text):
    scenes, scene, speech = [], [], None
    for block in text.split("\n\n"):
        lines = block.split("\n")
        if lines[0].startswith((" ", "[")):  # stage direction
            continue
        if SPEAKER.fullmatch(lines[0]) or lines[0].isupper() or lines[0].startswith(("SCENE", "Scene", "ACT", "Act")):
            if speech and speech[1]:
                scene.append(tuple(speech))
            speech = None
            if SPEAKER.fullmatch(lines[0]):
                speech = [lines[0][:-1], clean(lines[1:])]
                continue
            if len(scene) >= 2:  # a heading: the scene so far is done
                scenes.append(scene)
            scene = []
        elif speech is not None:  # verse resuming after a stage direction
            speech[1] = (speech[1] + "\n" + clean(lines)).strip()
    if speech and speech[1]:
        scene.append(tuple(speech))
    if len(scene) >= 2:
        scenes.append(scene)
    return scenes


scenes = split_scenes(text)
n_val = len(scenes) // 10
train_scenes, val_scenes = scenes[:-n_val], scenes[-n_val:]
print(f"{len(scenes)} scenes, {sum(map(len, scenes)):,} speeches; {len(val_scenes)} scenes held out")

balcony = next(s for s in scenes if any("yonder window" in speech for _, speech in s))
i = next(i for i, (_, speech) in enumerate(balcony) if "yonder window" in speech)
for name, speech in balcony[i:i + 4]:
    print(f"{name}.\n{speech[:150]}\n")

In [ ]:
# Same GPT as model.ipynb, with identical parameter names so v2's weights load as they are. Attention goes
# through F.scaled_dot_product_attention: at 1024 tokens the hand-written version is slow and keeps a
# (T, T) matrix per head for the backward pass. Generation feeds one token at a time against a KV cache.
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head, self.dropout = cfg.n_head, cfg.dropout
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.resid_drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_size)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        if cache is not None:
            # one new token at a time: it may see every cached position, so it needs no mask
            assert T == 1
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=cache is None, dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y)), (k, v)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x, cache=None):
        a, cache = self.attn(self.ln1(x), cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, cache


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layer))
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # tied embeddings

    def forward(self, idx, caches=None):
        B, T = idx.shape
        past = 0 if caches is None else caches[0][0].size(2)
        assert past + T <= self.cfg.block_size, "sequence longer than block_size"
        pos = torch.arange(past, past + T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        new_caches = []
        for i, block in enumerate(self.blocks):
            x, c = block(x, None if caches is None else caches[i])
            new_caches.append(c)
        return self.head(self.ln_f(x)), new_caches

In [ ]:
# Two models: v2 exactly as it is (for before/after comparisons), and the one to train: v2 with 1024 positions.
# Positions 0-255 keep v2's learned embeddings. The 768 new ones start as copies of positions 128-255, so they
# look like "far into the text" from the start, and training tells them apart. No dropout: v2 was trained
# without it, and switching it on for the fine-tune made the model worse in a trial run.
BLOCK_SIZE = 1024

base = GPT(GPTConfig(**v2_config)).to(device).eval()
base.load_state_dict({**v2_weights, "head.weight": v2_weights["tok_emb.weight"]})

cfg = GPTConfig(**{**v2_config, "block_size": BLOCK_SIZE})
pos = v2_weights["pos_emb.weight"]
extra = pos[128:].repeat(math.ceil((BLOCK_SIZE - len(pos)) / (len(pos) - 128)), 1)[:BLOCK_SIZE - len(pos)]
model = GPT(cfg).to(device)
model.load_state_dict({**v2_weights, "pos_emb.weight": torch.cat([pos, extra]), "head.weight": v2_weights["tok_emb.weight"]})

# check: within v2's 256 positions the grown model is still exactly v2
x = torch.tensor([encode("ROMEO.\nBut soft, what light through yonder window breaks?\n\nJULIET.\n")], device=device)
model.eval()
with torch.no_grad():
    assert torch.allclose(model(x)[0], base(x)[0], atol=1e-4)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params, context {cfg.block_size} tokens")

In [ ]:
# Training examples, in the chat page's format: "NAME.\nspeech\n\n" per speech, with the scene so far as context.
# - The speaker of the window's last speech plays "the model". The loss covers each of their speeches that
#   answers someone else, including the blank line that ends it, and nothing else: the model learns to
#   answer instead of continuing text. Their "NAME.\n" headers are context too; the chat page writes them.
# - Context is cut from the front at a speech boundary to fit. 30% of windows keep only 1-3 earlier
#   speeches, like the first turns of a chat. Half the time the line being answered is put on one line,
#   the way the chat page sends what you type.
# - The string always runs on into the next speaker's name (not trained on): a blank line before a name is
#   two "\n" tokens, but at the very end of a string it would be one "\n\n" token, which a chat never shows.
def dialogue_example(scenes, block_size, rng=random):
    """(token ids, loss mask over the targets ids[1:], characters under the loss)."""
    while True:
        scene = rng.choice(scenes)
        end = rng.randrange(1, len(scene))
        me = scene[end][0]
        if scene[end - 1][0] != me:
            break
    start = max(0, end - rng.randint(1, 3)) if rng.random() < 0.3 else 0
    flatten = rng.random() < 0.5
    after = scene[end + 1][0] if end + 1 < len(scene) else scene[end - 1][0]
    # skip ahead to the speeches that can fit (~3 characters per token here), then drop more until they do
    chars = 0
    for j in range(end, start - 1, -1):
        chars += len(scene[j][0]) + len(scene[j][1]) + 4
        if chars > 3.5 * block_size:
            start = j + 1
            break
    while start < end:
        pieces, spans, at = [], [], 0
        for j in range(start, end + 1):
            name, speech = scene[j]
            if flatten and j == end - 1:
                speech = " ".join(speech.split())
            head, body = f"{name}.\n", f"{speech}\n\n"
            if name == me and j > start and scene[j - 1][0] != me:
                spans.append((at + len(head), at + len(head) + len(body)))
            pieces.append(head + body)
            at += len(head) + len(body)
        enc = tok.encode("".join(pieces) + f"{after}.\n")
        if len(enc.ids) <= block_size + 1:
            break
        start += 1
    else:
        return dialogue_example(scenes, block_size, rng)  # the answer alone doesn't fit: pick another
    # a token is a target if it starts inside an answer; mask[t] is about predicting ids[t + 1]
    mask = [any(a <= s < b for a, b in spans) for s, _ in enc.offsets[1:]]
    return enc.ids, mask, sum(b - a for a, b in spans)


# plain Shakespeare windows (loss on every token), so the model keeps the language it already has
n = int(0.9 * len(text))
train_ids, val_ids = torch.tensor(encode(text[:n])), torch.tensor(encode(text[n:]))


def get_batch(split, batch_size, lm_rows):
    """lm_rows plain-text rows, the rest dialogue rows. Returns inputs, targets, loss mask, is-plain-text per row."""
    scenes_, ids_ = (train_scenes, train_ids) if split == "train" else (val_scenes, val_ids)
    rows = []
    for r in range(batch_size):
        if r < lm_rows:
            i = random.randrange(len(ids_) - BLOCK_SIZE - 1)
            ids = ids_[i:i + BLOCK_SIZE + 1].tolist()
            rows.append((ids, [True] * BLOCK_SIZE))
        else:
            rows.append(dialogue_example(scenes_, BLOCK_SIZE)[:2])
    T = max(len(ids) for ids, _ in rows) - 1
    x = torch.full((batch_size, T), EOT)
    y = torch.full((batch_size, T), EOT)
    mask = torch.zeros((batch_size, T), dtype=torch.bool)
    for r, (ids, m) in enumerate(rows):
        x[r, :len(ids) - 1] = torch.tensor(ids[:-1])
        y[r, :len(ids) - 1] = torch.tensor(ids[1:])
        mask[r, :len(m)] = torch.tensor(m)
    is_lm = torch.arange(batch_size) < lm_rows
    return x.to(device), y.to(device), mask.to(device), is_lm.to(device)

In [ ]:
# Checks: an example fits the context, has something to learn, and the trained tokens are exactly the
# answering character's speeches, each ending at its blank line. Trained text is shown in [[ ]].
for _ in range(200):
    ids, mask, n_chars = dialogue_example(train_scenes, BLOCK_SIZE)
    assert len(ids) <= BLOCK_SIZE + 1 and len(mask) == len(ids) - 1 and any(mask) and n_chars > 0
    targets = [t for t, m in zip(ids[1:], mask) if m]
    assert decode(targets).endswith("\n\n")

ids, mask, _ = dialogue_example(train_scenes, 160, rng=random.Random(3))
shown, inside = "", False
for t, m in zip(ids[1:], mask):
    if m != inside:
        shown += "[[" if m else "]]"
        inside = m
    shown += decode([t])
print(decode(ids[:1]) + shown + ("]]" if inside else ""))

x, y, mask, is_lm = get_batch("train", 4, 1)
assert torch.equal(x[0, 1:], y[0, :-1])  # the plain-text row: targets are inputs shifted by one
assert mask[0].all() and not mask[1:].all() and is_lm.tolist() == [True, False, False, False]
print("all checks passed")

In [ ]:
# How well a model answers, measured on the held-out scenes. v2 may have seen them in its own training, so
# these numbers compare what the models were trained to do, not how well they generalise.
# - reply loss: nats per character of the answering character's speeches, given the context the chat page
#   fits in 256 tokens today, and given 1024 tokens;
# - line gain: how much more likely the real reply is after the line it actually answered than after a
#   random line from elsewhere. This is the test that showed v2 barely listens (~2%).
EVAL_EXAMPLES, EVAL_PAIRS = 300, 300
use_amp = device == "cuda"
amp = lambda: torch.autocast("cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()

eval_256 = [dialogue_example(val_scenes, 256, rng=random.Random(i)) for i in range(EVAL_EXAMPLES)]
eval_1024 = [dialogue_example(val_scenes, BLOCK_SIZE, rng=random.Random(i)) for i in range(EVAL_EXAMPLES)]

pairs = [(a, b) for s in val_scenes for a, b in zip(s, s[1:]) if a[0] != b[0] and 30 <= len(a[1]) <= 250 and 30 <= len(b[1]) <= 250]
pairs = random.Random(0).sample(pairs, min(EVAL_PAIRS, len(pairs)))


@torch.no_grad()
def nats(m, ids, mask):
    m.eval()
    x = torch.tensor([ids[:-1]], device=device)
    with amp():
        logits = m(x)[0][0]
    ce = F.cross_entropy(logits.float(), torch.tensor(ids[1:], device=device), reduction="none")
    return ce[torch.tensor(mask, device=device)].sum().item()


def reply_loss(m, examples):
    return sum(nats(m, ids, mask) for ids, mask, _ in examples) / sum(c for _, _, c in examples)


def line_gain(m):
    def score(said, a, b):
        enc = tok.encode(f"{a[0]}.\n{said}\n\n{b[0]}.\n" + f"{b[1]}\n\n" + f"{a[0]}.\n")
        first = len(f"{a[0]}.\n{said}\n\n{b[0]}.\n")
        last = first + len(b[1]) + 2
        return nats(m, enc.ids, [first <= s < last for s, _ in enc.offsets[1:]])

    real = sum(score(a[1], a, b) for a, b in pairs)
    other = sum(score(pairs[(i + 1) % len(pairs)][0][1], a, b) for i, (a, b) in enumerate(pairs))
    return 100 * (other - real) / other


def report(name, m):
    row = {"reply loss @256": reply_loss(m, eval_256), "line gain %": line_gain(m)}
    if m.cfg.block_size >= BLOCK_SIZE:
        row["reply loss @1024"] = reply_loss(m, eval_1024)
    print(f"{name:22s} " + " | ".join(f"{k} {v:.3f}" for k, v in row.items()))
    return row


before = report("v2", base)
report("v2, 1024 positions", model)  # untrained new positions: worse past 256 tokens until training

In [ ]:
# Fine-tuning config. A batch is 16 rows of up to 1024 tokens: 4 rows of plain Shakespeare (loss on every
# token, weight 0.2) and 12 dialogue rows (loss on the answers, weight 0.8).
# The learning rate is 20x below model.ipynb's from-scratch 1e-3: in a trial run, 3e-4 knocked the trained
# model off course and its answers got worse. The new position embeddings start untrained, so they learn
# 10x faster than the rest. Warmup, then cosine decay.
# fp16 autocast + gradient scaling on CUDA only. About 15-20 minutes on a Colab T4.
batch_size = 16
lm_rows = 4
lm_weight = 0.2
max_iters = 3000
lr, min_lr = 5e-5, 5e-6
pos_lr_scale = 10
warmup_iters = 100
grad_clip = 1.0
eval_interval = 250
eval_iters = 20

scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def get_lr(it):
    if it < warmup_iters:
        return lr * (it + 1) / warmup_iters
    progress = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * progress))


def losses(logits, y, mask, is_lm):
    """(training loss, mean loss on dialogue answers, mean loss on plain text); a missing part is nan."""
    ce = F.cross_entropy(logits.float().flatten(0, 1), y.flatten(), reduction="none").view_as(y)
    reply = ce[mask & ~is_lm[:, None]].mean()
    plain = ce[mask & is_lm[:, None]].mean()
    if not is_lm.any():
        return reply, reply, plain
    if is_lm.all():
        return plain, reply, plain
    return (1 - lm_weight) * reply + lm_weight * plain, reply, plain


@torch.no_grad()
def estimate_loss(m):
    m.eval()
    out = {}
    for split in ("train", "val"):
        r, p = torch.zeros(eval_iters), torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y, mask, is_lm = get_batch(split, batch_size, lm_rows)
            with amp():
                logits = m(x)[0]
            _, r[k], p[k] = losses(logits, y, mask, is_lm)
        out[split] = (r.mean().item(), p.mean().item())
    m.train()
    return out

In [ ]:
# Training loop: AdamW, gradient clipping, loss on answers and on plain text every eval_interval steps.
rest = [p for name, p in model.named_parameters() if name != "pos_emb.weight"]
optimizer = torch.optim.AdamW(
    [{"params": rest, "scale": 1}, {"params": [model.pos_emb.weight], "scale": pos_lr_scale}], lr=lr, betas=(0.9, 0.99)
)
model.train()

log = []
t0 = time.time()
for it in range(max_iters):
    for group in optimizer.param_groups:
        group["lr"] = get_lr(it) * group["scale"]
    if it % eval_interval == 0 or it == max_iters - 1:
        est = estimate_loss(model)
        log.append((it, *est["train"], *est["val"]))
        print(f"step {it:5d} | answers: train {est['train'][0]:.4f} val {est['val'][0]:.4f} | "
              f"plain text: train {est['train'][1]:.4f} val {est['val'][1]:.4f} | {time.time() - t0:.0f}s")

    x, y, mask, is_lm = get_batch("train", batch_size, lm_rows)
    with amp():
        logits = model(x)[0]
    loss = losses(logits, y, mask, is_lm)[0]
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

model.eval()
print(f"done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Loss on answers (what this fine-tune is for) and on plain text (what it should keep), per token.
import matplotlib.pyplot as plt

steps, ans_train, txt_train, ans_val, txt_val = zip(*log)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, tr, va, title in ((ax[0], ans_train, ans_val, "answers"), (ax[1], txt_train, txt_val, "plain text")):
    a.plot(steps, tr, label="train")
    a.plot(steps, va, label="val")
    a.set_xlabel("step")
    a.set_ylabel("cross-entropy loss")
    a.set_title(title)
    a.legend()
plt.show()

In [ ]:
# Before and after, on the same held-out examples. Lower reply loss is better; higher line gain means the
# reply depends more on what was said to it. The number that matters for the chat is v3 with its 1024 tokens
# against v2 with the 256 the chat page gives it today.
after = report("v3", model)
print()
print(f"answers, as each model is served: v2 @256 {before['reply loss @256']:.3f} -> v3 @1024 {after['reply loss @1024']:.3f} nats/char")
print(f"answers with the same 256 tokens: {before['reply loss @256']:.3f} -> {after['reply loss @256']:.3f}")
print(f"line gain:                        {before['line gain %']:.1f}% -> {after['line gain %']:.1f}%")

In [ ]:
# Play scenes the way the chat page does: the scene so far written as a script and cut from the front to
# fit, "NAME.\n" for the model's turn, temperature 0.7 and top-k 40, a reply ends at its first blank line.
# v2 gets the chat page's budget for 256 tokens, v3 the budget for 1024.
SCENES = [
    ("ROMEO", "JULIET", "But soft, what light through yonder window breaks?"),
    ("HAMLET", "OPHELIA", "Lady, shall I lie in your lap?"),
    ("PROSPERO", "MIRANDA", "Thy father was the Duke of Milan."),
    ("LEAR", "CORDELIA", "What can you say to draw a third more opulent than your sisters?"),
]
LINES = ["Dost thou love me? Answer plainly.", "Why dost thou speak so strangely?", "Tell me, what shall we do tomorrow?"]
MAX_TOKENS, CHARS_PER_TOKEN = 40, 2.5  # the chat page's settings for word-piece models


def build_prompt(lines, nxt, budget):
    script = "".join(f"{s}.\n{t}\n\n" for s, t in lines) + f"{nxt}.\n"
    if len(script) <= budget:
        return script
    tail = script[-budget:]
    return tail[tail.index("\n\n") + 2:]


@torch.no_grad()
def reply(m, prompt, temperature=0.7, top_k=40):
    ids = encode(prompt)
    logits, caches = m(torch.tensor([ids], device=device))
    out = []
    for _ in range(MAX_TOKENS):
        logits = logits[0, -1].float() / temperature
        logits[logits < torch.topk(logits, top_k).values[-1]] = -float("inf")
        t = torch.multinomial(F.softmax(logits, dim=-1), 1).item()
        if t == EOT:
            break
        out.append(t)
        if "\n\n" in decode(out) or len(ids) + len(out) >= m.cfg.block_size:
            break
        logits, caches = m(torch.tensor([[t]], device=device), caches=caches)
    text = decode(out)
    if "\n\n" in text:
        return text[:text.index("\n\n")]
    cut = re.match(r"^[\s\S]*[.!?]", text)  # out of length mid-sentence: end at the last full sentence
    return cut.group(0) if cut else text


torch.manual_seed(0)
for you, them, opening in SCENES:
    print("=" * 70)
    for name, m in (("v2", base), ("v3", model)):
        budget = math.floor((m.cfg.block_size - MAX_TOKENS) * CHARS_PER_TOKEN)
        lines = []
        print(f"--- {name}")
        for said in [opening] + LINES:
            answer = reply(m, build_prompt(lines + [(you, said)], them, budget))
            lines += [(you, said), (them, answer)]
            print(f"{you}: {said}\n    {them}: {answer!r}")

In [ ]:
# Save in the inference server's format: the npz holds the weights (head.weight is tok_emb.weight, so it is
# left out), the config and the tokenizer. The torch checkpoint is kept for later fine-tunes, plus the loss log.
weights = {k: v.detach().float().cpu().numpy() for k, v in model.state_dict().items() if k != "head.weight"}
np.savez("dialogue.npz", **weights, config=np.array(json.dumps(asdict(cfg))), tokenizer=np.array(tokenizer_json))
torch.save({"model": model.state_dict(), "config": asdict(cfg)}, "dialogue.pt")
with open("losses.csv", "w") as f:
    f.write("step,answers_train,text_train,answers_val,text_val\n")
    f.writelines(",".join(f"{v:.4f}" if isinstance(v, float) else str(v) for v in row) + "\n" for row in log)
print(f"dialogue.npz: {Path('dialogue.npz').stat().st_size / 1e6:.1f} MB")

In [ ]:
# Check the export with the inference server's own numpy code (server/gpt.py from GitHub): it loads the npz
# the way the server will, its logits match torch, and greedy decoding writes the same text. Then the
# fine-tune itself: with its longer context v3's answers are more likely than v2's as served today, and they
# are no worse when both get the same 256 tokens.
urllib.request.urlretrieve("https://raw.githubusercontent.com/hr5ht/tinygptinferenceserver/main/server/gpt.py", "server_gpt.py")
import server_gpt

with np.load("dialogue.npz") as f:
    w = {k: f[k] for k in f.files}
served_tok = server_gpt.BPETokenizer(str(w.pop("tokenizer")))
served = server_gpt.GPT(server_gpt.GPTConfig(**json.loads(str(w.pop("config")))), w, served_tok.eot)

prompt = "ROMEO.\nBut soft, what light through yonder window breaks?\n\nJULIET.\n"
ids = served_tok.encode(prompt)
assert ids == encode(prompt)
with torch.no_grad():
    torch_logits = model(torch.tensor([ids], device=device))[0][0].float().cpu().numpy()
numpy_logits = served(np.array([ids]))[0][0]
assert np.abs(torch_logits - numpy_logits).max() < 1e-3, np.abs(torch_logits - numpy_logits).max()

served_greedy = [t for _, t, _ in server_gpt.generate_batch(served, [ids], [60], [{"temperature": 0}])]
torch_greedy, idx = [], torch.tensor([ids], device=device)
with torch.no_grad():
    for _ in range(60):
        t = model(idx)[0][0, -1].argmax().item()
        torch_greedy.append(t)
        if t == EOT:
            break
        idx = torch.cat([idx, torch.tensor([[t]], device=device)], dim=1)
assert served_greedy == torch_greedy

assert after["reply loss @1024"] < before["reply loss @256"], "the longer context did not make answers more likely"
assert after["reply loss @256"] < before["reply loss @256"] + 0.005, "answers got worse with the same context"

print(prompt + decode(served_greedy))
print("=" * 60)
print("TRAINING SUCCESSFUL")
print(f"answers: v2 @256 {before['reply loss @256']:.3f} -> v3 @{cfg.block_size} {after['reply loss @1024']:.3f} nats/char | "
      f"line gain {before['line gain %']:.1f}% -> {after['line gain %']:.1f}%")
print("the server's numpy code loads dialogue.npz and generates the same text as the trained model")
print("=" * 60)

In [ ]:
# Upload to the Hugging Face Hub as a new model repo, next to v1 and v2. The server downloads dialogue.npz.
# The token is read with getpass, so it is never stored in the notebook.
import getpass
from huggingface_hub import HfApi

api = HfApi(token=getpass.getpass("HF write token: "))
repo_id = f"{api.whoami()['name']}/tinygpt-shakespeare-v3"
api.create_repo(repo_id, exist_ok=True)
for f in ("dialogue.npz", "dialogue.pt", "losses.csv"):
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=repo_id)
print(f"uploaded to https://huggingface.co/{repo_id}")